In [3]:
import requests

base_faq_url = 'https://datatalks.club/faq'
courses_index_url = f'{base_faq_url}/json/courses.json'

courses_index = requests.get(courses_index_url).json()
courses_index

[{'course': 'ai-dev-tools-zoomcamp',
  'course_name': 'AI Dev Tools Zoomcamp',
  'path': '/json/ai-dev-tools-zoomcamp.json',
  'questions_count': 48},
 {'course': 'machine-learning-zoomcamp',
  'course_name': 'ML Zoomcamp',
  'path': '/json/machine-learning-zoomcamp.json',
  'questions_count': 460},
 {'course': 'data-engineering-zoomcamp',
  'course_name': 'Data Engineering Zoomcamp',
  'path': '/json/data-engineering-zoomcamp.json',
  'questions_count': 397},
 {'course': 'llm-zoomcamp',
  'course_name': 'LLM Zoomcamp',
  'path': '/json/llm-zoomcamp.json',
  'questions_count': 153},
 {'course': 'stock-markets-analytics-zoomcamp',
  'course_name': 'Stock Markets Analytics Zoomcamp',
  'path': '/json/stock-markets-analytics-zoomcamp.json',
  'questions_count': 93},
 {'course': 'mlops-zoomcamp',
  'course_name': 'MLOps Zoomcamp',
  'path': '/json/mlops-zoomcamp.json',
  'questions_count': 249}]

In [4]:
course_url = f'{base_faq_url}/json/llm-zoomcamp.json'
course_data = requests.get(course_url).json()

course_data[:3]

[{'id': '74eb249bbf',
  'course': 'llm-zoomcamp',
  'section': 'General Course-Related Questions',
  'question': 'I just discovered the course. Can I still join?',
  'answer': 'Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions.'},
 {'id': '977bf7786c',
  'course': 'llm-zoomcamp',
  'section': 'General Course-Related Questions',
  'question': 'Course: I have registered for the LLM Zoomcamp. When can I expect to receive the confirmation email?',
  'answer': "You don't need it. You're accepted. You can also just start learning and submitting homework (while the form is open) without registering. It is not checked against any registered list. Registration is just to gauge interest before the start date."},
 {'id': '489dd1c9d9',
  'course': 'llm-zoomcamp',
  'section': 'General Course-Related Questions',
  'question': 'What is the video/zoom link to the stream for the “Office Hours” or live/workshop sessions?',
  'answer':

In [5]:
documents = []

for course in courses_index:
    course_path = course['path']
    course_url = f'{base_faq_url}/{course_path}'
    course_data = requests.get(course_url).json()
    documents.extend(course_data)

len(documents)

1400

In [6]:
from minsearch import Index

index = Index(
    text_fields=['section', 'question', 'answer'],
    keyword_fields=['course']
)

index.fit(documents)

In [7]:
question = 'I just discovered the course. Can I join now?'

filter_dict = {
    'course': 'llm-zoomcamp'
}

boost_dict = {
    'question': 3,
    'section': 0.5,
}

search_results = index.search(
    question,
    filter_dict=filter_dict,
    boost_dict=boost_dict,
    num_results=5
)

In [8]:
search_results

[{'id': '74eb249bbf',
  'course': 'llm-zoomcamp',
  'section': 'General Course-Related Questions',
  'question': 'I just discovered the course. Can I still join?',
  'answer': 'Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions.'},
 {'id': '193612db63',
  'course': 'llm-zoomcamp',
  'section': 'Module 3: Orchestration',
  'question': "Why do we need orchestration / Kestra — can't I just run the code in a notebook?",
  'answer': "Notebooks are great for learning and experimenting, but real AI workflows need more than a script that runs once: scheduling, retries, monitoring, secret management, and reliably chaining tasks together. That's what an orchestrator like Kestra provides.\n\nIn this module Kestra is also the vehicle for the AI techniques the course is teaching: AI Copilot to generate flows from natural language, RAG to ground responses in real data, and AI agents that decide which tools to call. The goal is to se

In [9]:
def search(question):
    filter_dict = {
        'course': 'llm-zoomcamp'
    }

    boost_dict = {
        'question': 3,
        'section': 0.5,
    }

    search_results = index.search(
        question,
        filter_dict=filter_dict,
        boost_dict=boost_dict,
        num_results=5
    )

    return search_results

In [10]:
import rag
from openai import OpenAI

openai_client = OpenAI()

In [11]:
faq_rag = rag.RAG(
    index=index,
    llm_client=openai_client
)

In [12]:
answer = faq_rag.rag(question)

print(answer.answer)

Yes, you can still join the course. However, if you want to receive a certificate, you need to submit your project while submissions are still being accepted.


In [13]:
class LLMZoomcampFAQRAG(rag.RAG):

    def search(self, query):
        print('using search from LLMZoomcampFAQRAG...')

        filter_dict = {
            'course': 'llm-zoomcamp'
        }

        boost_dict = {
            'question': 3,
            'section': 0.5,
        }

        search_results = self.index.search(
            query,
            filter_dict=filter_dict,
            boost_dict=boost_dict,
            num_results=5
        )

        return search_results

In [14]:
faq_rag = LLMZoomcampFAQRAG(
    index=index,
    llm_client=openai_client
)


In [15]:
answer = faq_rag.rag(question)
print(answer.answer)

using search from LLMZoomcampFAQRAG...
Yes, you can still join the course. However, if you want to receive a certificate, you need to submit your project while submissions are still being accepted.


In [16]:
from typing import Literal
from pydantic import BaseModel, Field


class Reference(BaseModel):
    document_id: str = Field(description="Reference to FAQ 'id' field that contains the answer")


class FAQRAGResponse(BaseModel):
    """
    This model provides a structured answer with metadata about the response,
    including confidence, categorization, and follow-up suggestions.
    """

    answer: str = Field(description="The main answer to the user's question in markdown")
    found_answer: bool = Field(description="True if relevant information was found in the documentation")
    confidence: float = Field(description="Confidence score from 0.0 to 1.0 indicating how certain the answer is")
    confidence_explanation: str = Field(description="Explanation about the confidence level")
    answer_type: Literal["how-to", "explanation", "troubleshooting", "comparison", "reference"] = Field(description="The category of the answer")
    followup_questions: list[str] = Field(description="Suggested follow-up questions the user might want to ask")

    references: list[Reference]

In [17]:
faq_rag = LLMZoomcampFAQRAG(
    index=index,
    llm_client=openai_client,
    output_type=FAQRAGResponse
)

In [18]:
answer = faq_rag.rag(question)

print(answer.answer)
print(answer.references)

using search from LLMZoomcampFAQRAG...
Yes, you can still join the course. However, if you want to receive a certificate, you need to submit your project while submissions are still being accepted.
[Reference(document_id='74eb249bbf')]


In [19]:
reference_url_template = 'https://datatalks.club/faq/{course_name}.html#{faq_id}'

reference_url_template.format(
    course_name='llm-zoomcamp',
    faq_id='74eb249bbf'
)

'https://datatalks.club/faq/llm-zoomcamp.html#74eb249bbf'